# Plate detection — A1 (YOLO26n vs YOLOv8n) — Kaggle
Thin driver (OS-independent). All logic lives in `plate_detect`; this notebook only calls the CLI.

**Kaggle runtime only.** Setup clones the private repo, installs the CLI, and points raw A1 at the attached Kaggle Dataset. All later cells run from the repo root at `/kaggle/working`.

**Before running:**
1. Notebook settings → **Internet: ON** (git clone + pip).
2. Notebook settings → **Accelerator: GPU** (T4×2 or P100).
3. Add-ons → **Secrets** → add `GH_PAT` (GitHub PAT with repo read).
4. **Add data** → search `duydieunguyen/licenseplates` → adds it at `/kaggle/input/licenseplates`.

In [ ]:
# === Setup (Kaggle) ===
# Clones private repo BRANCH into /kaggle/working, installs CLI, symlinks raw A1 to the attached dataset.
import os, glob, shutil, socket
from kaggle_secrets import UserSecretsClient

assert os.path.isdir("/kaggle"), "this notebook is Kaggle-only"

# --- fail early, with a clear reason, before any slow work ---
try:
    socket.create_connection(("github.com", 443), timeout=5).close()
except OSError:
    raise SystemExit(
        "Internet is OFF on this kernel. Settings -> Internet: On "
        "(requires phone verification: kaggle.com/settings). Enable, then re-run."
    )
assert os.path.isdir("/kaggle/input/licenseplates"), (
    "dataset not attached. Add data -> duydieunguyen/licenseplates, then re-run."
)
try:
    tok = UserSecretsClient().get_secret("GH_PAT")
except Exception:
    raise SystemExit("Secret GH_PAT missing. Add-ons -> Secrets -> GH_PAT = GitHub PAT (repo read), then re-run.")

REPO   = "/kaggle/working/UIT2026-DoAnCuoiKi"
BRANCH = "feat/plate-detect-a1"                 # package NOT merged to main yet — clone this branch
RAW    = "data/raw/kaggle_vn_plate_segment"     # layout the A1Adapter expects: {images,labels}/{train,val}
KDATA  = "/kaggle/input/licenseplates"          # attached dataset: kaggle.com/datasets/duydieunguyen/licenseplates

# --- clone private repo, feature branch (PAT from GH_PAT above) ---
if not os.path.isdir(REPO):
    !git clone --branch {BRANCH} --single-branch https://{tok}@github.com/UIT-DoAnCuoiKi/UIT2026-DoAnCuoiKi.git {REPO}
del tok
assert os.path.isdir(REPO), "clone failed — PAT lacks read access to the org repo (see README 'PAT setup')"
%cd {REPO}
!git rev-parse --abbrev-ref HEAD   # confirm the feature branch is checked out

# --- install the package -> puts the `plate_detect` CLI on PATH ---
!pip install -q -e src/ml/plate_detection_pipeline

# --- symlink RAW -> the data root under the attached dataset that holds images/train ---
if not os.path.isdir(f"{RAW}/images/train"):
    hits = glob.glob(f"{KDATA}/**/images/train", recursive=True)
    assert hits, f"images/train not found under {KDATA} — check the dataset layout"
    root = os.path.abspath(hits[0][: -len("/images/train")])
    os.makedirs(os.path.dirname(RAW), exist_ok=True)
    if os.path.islink(RAW) or os.path.exists(RAW):
        (os.unlink if os.path.islink(RAW) else shutil.rmtree)(RAW)
    os.symlink(root, os.path.abspath(RAW))
    print("raw A1 ->", root)

# sanity-check the raw layout the adapter reads (train + val, images + labels)
for s in ("train", "val"):
    for k in ("images", "labels"):
        assert os.path.isdir(f"{RAW}/{k}/{s}"), f"missing {RAW}/{k}/{s} — check split names (val vs valid)"
print("OK — CLI installed, raw A1 ready at", RAW)

In [ ]:
import torch; print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Prepare (class-map gate → split → dedup train↔test & train↔val → validate)

In [ ]:
!plate_detect prepare

In [ ]:
!plate_detect check

## 2. Train — full matrix @640 (both models × seeds 0,1,2)

In [ ]:
# QUICK SMOKE TEST (10 epochs, 1 seed via configs/quick.yaml). For real run drop --config and use: --imgsz 640 --seeds 0,1,2
!plate_detect train --config src/ml/plate_detection_pipeline/configs/quick.yaml --project runs

## 3. imgsz ablation @960 (single seed, both models)

In [ ]:
# QUICK SMOKE TEST (10 epochs, 1 seed). --imgsz 960 overrides quick.yaml. For real run drop --config and use: --imgsz 960 --seeds 0
!plate_detect train --config src/ml/plate_detection_pipeline/configs/quick.yaml --imgsz 960 --project runs

## 4. Export best → ONNX (per model & imgsz), parity-checked

In [ ]:
# example; repeat per model/imgsz best run:
!plate_detect export --weights runs/yolo26n_s0_640/weights/best.pt --out weights/yolo26n_a1_640.onnx --imgsz 640

## 5. Evaluate on A1 test → comparison table + experiments.csv

In [ ]:
!plate_detect eval --imgszs 640,960 --project runs --weights-dir weights --sample-image data/processed/a1_det/images/test/$(ls data/processed/a1_det/images/test | head -1)

## 6. Package results

Zips `runs/`, `weights/`, and `experiments.csv` into one archive under `/kaggle/working/`. Kaggle persists `/kaggle/working` automatically — the zip appears in the notebook **Output** tab, downloadable after the session. No Drive mount needed.

In [ ]:
# === Package all results into one .zip under /kaggle/working ===
import os, datetime

STAMP   = datetime.datetime.now().strftime("%Y%m%d_%H%M")
ARCHIVE = f"/kaggle/working/plate_det_results_{STAMP}.zip"

# collect what exists (runs/ = weights+plots+curves, weights/ = exported ONNX, experiments.csv)
targets = [p for p in ("runs", "weights", "experiments.csv") if os.path.exists(p)]
assert targets, "nothing to zip — run train/export/eval first"
print("zipping:", targets)
!zip -rq "{ARCHIVE}" {" ".join(targets)}
print("archive:", ARCHIVE, f"({os.path.getsize(ARCHIVE)/1e6:.1f} MB)")
print("grab it from the notebook Output tab after the session ends.")